# Leakage score gap: what a leaky pipeline reports vs what production sees

[`09_leakage_safety.ipynb`](09_leakage_safety.ipynb) shows *how* skyulf blocks learned
preprocessing before a split. This notebook measures *how much it matters*.

We train the **same logistic-regression model** twice on the same rows:

- **Leaky:** `UnivariateSelection -> TrainTestSplitter -> model`. The feature selector
  learns which columns matter from **all** rows, including the test rows it is later
  graded on. Run on purpose with `on_leakage="ignore"`.
- **Safe:** `TrainTestSplitter -> UnivariateSelection -> model`, the order the leakage
  check asks for.

For each pipeline we compare the held-out test **accuracy skyulf reports** with its
accuracy on **5,000 fresh rows** drawn from the same process ("production").

The data is synthetic, seeded and built in memory: 300 rows, 2,000 numeric features,
of which only 10 carry signal. This wide, small-sample shape is chosen on purpose,
because it is where selection leakage bites (gene panels, sensor logs, wide customer
tables). The experiment repeats over 10 seeds and runs in about 15 seconds.
The terminal version is [`10_leakage_score_gap.py`](10_leakage_score_gap.py).

In [1]:
import logging
from copy import deepcopy
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score

from skyulf import SkyulfPipeline, validate_leakage_safety
from skyulf.leakage import OnLeakage

N_ROWS = 300
N_FEATURES = 2000
N_INFORMATIVE = 10
N_SELECTED = 30
N_PRODUCTION_ROWS = 5000
SEEDS = range(10)


def _require(condition: bool, message: str) -> None:
    """Keep tutorial correctness checks active under optimized Python."""
    if not condition:
        raise RuntimeError(message)


def make_dataset(rng: np.random.Generator, n_rows: int) -> pd.DataFrame:
    """Draw rows where only the first N_INFORMATIVE features drive the label.

    P(target = 1) = sigmoid(f0000 + ... + f0009); the other features are pure noise.
    """
    features = rng.normal(size=(n_rows, N_FEATURES))
    probability = 1.0 / (1.0 + np.exp(-features[:, :N_INFORMATIVE].sum(axis=1)))
    data = pd.DataFrame(features, columns=[f"f{i:04d}" for i in range(N_FEATURES)])
    data["target"] = (rng.random(n_rows) < probability).astype(int)
    return data


logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")

## 1. Two orders of the same steps

`UnivariateSelection` keeps the 30 columns with the highest ANOVA F-score against
the target. That score is **learned from labels**, so the rows it sees must not
include the rows used to grade the model.

In [2]:
SPLIT_STEP = {
    "name": "split",
    "transformer": "TrainTestSplitter",
    "params": {"test_size": 0.3, "random_state": 42, "target_column": "target", "stratify": True},
}
# Keeps the 30 columns with the highest ANOVA F-score against the target.
SELECT_STEP = {
    "name": "select",
    "transformer": "UnivariateSelection",
    "params": {"method": "SelectKBest", "k": N_SELECTED, "target_column": "target"},
}
MODEL = {"type": "logistic_regression"}
LEAKY_CONFIG = {"preprocessing": [SELECT_STEP, SPLIT_STEP], "modeling": MODEL}
SAFE_CONFIG = {"preprocessing": [SPLIT_STEP, SELECT_STEP], "modeling": MODEL}

## 2. The leakage check flags one and passes the other

`validate_leakage_safety()` inspects the config only; it does not train anything.
With the default `on_leakage="raise"`, `fit()` refuses the leaky order outright.

In [3]:
leaky_messages = validate_leakage_safety(LEAKY_CONFIG, on_leakage="warn")
_require(bool(leaky_messages), "Selection before the split must be flagged")
_require(validate_leakage_safety(SAFE_CONFIG) == [], "The split-first config must pass")
display({"leaky": leaky_messages, "safe": validate_leakage_safety(SAFE_CONFIG)})

try:
    SkyulfPipeline(deepcopy(LEAKY_CONFIG)).fit(
        make_dataset(np.random.default_rng(0), 50), target_column="target"
    )
except ValueError:
    print("With the default on_leakage='raise', fit() refuses the leaky order.")
else:
    raise AssertionError("The default policy must reject the leaky order")

{'leaky': ["Step 0 ('UnivariateSelection') is configured before the train/test split (step 1, 'TrainTestSplitter') and fits its statistics on the full dataset including the test set - move it after the splitter."],
 'safe': []}

With the default on_leakage='raise', fit() refuses the leaky order.


## 3. Train both, then score on fresh rows

For each seed we draw a 300-row training table and a separate 5,000-row production
table from the same process. `reported` is the test-split accuracy from
`fit()`'s metrics; `production` is `accuracy_score` on the fresh rows.
The leaky order runs with `on_leakage="ignore"` (section 2 already showed its warning);
the safe order runs under the default `"raise"` policy.

In [4]:
def reported_and_production(
    config: dict[str, Any], data: pd.DataFrame, production: pd.DataFrame, on_leakage: OnLeakage
) -> tuple[float, float]:
    """Fit once; return (held-out accuracy skyulf reports, accuracy on fresh rows)."""
    pipeline = SkyulfPipeline(deepcopy(config))
    metrics = pipeline.fit(data, target_column="target", on_leakage=on_leakage)
    reported = metrics["modeling"]["splits"]["test"].metrics["accuracy"]
    predictions = pipeline.predict(production.drop(columns="target"))
    return float(reported), float(accuracy_score(production["target"], predictions))


rows = []
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    data = make_dataset(rng, N_ROWS)
    production = make_dataset(rng, N_PRODUCTION_ROWS)
    leaky = reported_and_production(LEAKY_CONFIG, data, production, "ignore")
    safe = reported_and_production(SAFE_CONFIG, data, production, "raise")
    rows.append(
        {
            "seed": seed,
            "leaky_reported": leaky[0],
            "leaky_production": leaky[1],
            "safe_reported": safe[0],
            "safe_production": safe[1],
            "base_rate": production["target"].mean(),
        }
    )
results = pd.DataFrame(rows).set_index("seed")
display(results.round(3))

,leaky_reported,leaky_production,safe_reported,safe_production,base_rate
seed,,,,,
0,0.933,0.760,0.689,0.667,0.490
1,0.933,0.765,0.744,0.742,0.480
2,0.889,0.755,0.711,0.712,0.506
3,0.878,0.729,0.811,0.685,0.490
4,0.911,0.770,0.756,0.730,0.514
5,0.911,0.735,0.778,0.630,0.500
6,0.778,0.731,0.644,0.686,0.501
7,0.867,0.774,0.689,0.705,0.503
8,0.900,0.770,0.689,0.671,0.497


## 4. The gap, averaged over seeds

A 90-row test split has a standard error of about 5 points, and single seeds
above are off by up to 15 points in either pipeline. That is why the 10-seed mean
is the number to read; the checks below pin it so it cannot drift silently.

In [5]:
mean = results.mean()
leaky_gap = mean["leaky_reported"] - mean["leaky_production"]
safe_gap = mean["safe_reported"] - mean["safe_production"]
summary = pd.DataFrame(
    {
        "reported (held-out test)": [mean["leaky_reported"], mean["safe_reported"]],
        "production (fresh rows)": [mean["leaky_production"], mean["safe_production"]],
        "gap": [leaky_gap, safe_gap],
    },
    index=["leaky: select -> split", "safe: split -> select"],
)
print(f"Mean accuracy (%) over {len(SEEDS)} seeds; positive-class rate {mean['base_rate']:.1%}")
display((summary * 100).round(1))

_require(
    leaky_gap >= 0.10,
    f"The leaky report must overstate production by >= 10 points: {leaky_gap:.3f}",
)
_require(
    abs(safe_gap) <= 0.05,
    f"The safe report must match production within 5 points: {safe_gap:.3f}",
)

Mean accuracy (%) over 10 seeds; positive-class rate 49.8%


,reported (held-out test),production (fresh rows),gap
leaky: select -> split,88.4,74.5,13.9
safe: split -> select,71.6,69.0,2.5


## 5. Reading the result

- **The leaky report is wrong by more than 10 points.** The selector saw the test rows'
  labels, so among 2,000 noise columns it kept some that happen to predict exactly those
  rows. The held-out score was graded against its own answer key.
- **The safe report predicts production.** Its gap is within sampling noise.
- **The leaky *model* is not worse here.** Picking features from all 300 rows finds the
  real ones slightly more often, so its production accuracy can even be higher. The
  defect is the **reported score**: you would ship believing the wrong number.
- **Remedy:** report the split-first score. If you then want a model trained on every
  row, retrain after recording that estimate, and quote the split-first number.

The same mechanism applies to any step that learns from labels or data statistics
before the split; see [`09_leakage_safety.ipynb`](09_leakage_safety.ipynb) and the
[preprocessing placement reference](../../docs/user_guide/preprocessing_placement.md).